# 🐔 WWT Unravel 2025 – Recommendation Algorithm for Wings R Us
**Team:** DisCovert

## 📌 Introduction
This notebook contains the full data exploration and preprocessing pipeline for the **WWT Unravel 2025** case competition.

**Client:** Wings R Us – a US-based quick service restaurant (QSR) specializing in chicken wings, sides, and beverages.

**Goal:** Build a **personalized last-minute item recommendation system** for the checkout stage to:
- Increase average order value (AOV)
- Improve customer retention
- Keep recommendations fresh and relevant

## 📄 Problem Statement (From Case File)

Wings R Us wants to enhance its **digital checkout experience** by introducing **smart, last-minute recommendations** across its app, website, and kiosks.

### Business Notes:
- Use available data to identify **customer signals** that drive recommendations
- Serve a **diverse customer base** (first-timers, loyal customers, special members)
- Recommendations should **avoid repetition** and feel **fresh**
- Define and measure success — metrics like **Recall@3**

### Provided Datasets:
1. **order_data** – 1.4M orders with customer, store, channel, and order details (JSON format for items)
2. **customer_data** – 550K customers with type classification
3. **store_data** – 38 stores with city, state, postal info
4. **test_data_question** – 1K sample carts with one item missing for scoring

### Step 1: Code execution

In [ ]:
#pip install tqdm

### Step 2: Code execution

In [ ]:
# pip install tqdm

import pandas as pd
import numpy as np
import json
from tqdm import tqdm

### Step 3: Load datasets into DataFrames

In [ ]:
# Load the main datasets
order_data = pd.read_csv('order_data.csv')
customer_data = pd.read_csv('customer_data.csv')
store_data = pd.read_csv('store_data.csv')
test_data = pd.read_csv('test_data_question.csv')

### Step 4: Code execution

In [ ]:
# Display basic info about each dataset
print("Order Data Shape:", order_data.shape)
print("Customer Data Shape:", customer_data.shape)
print("Store Data Shape:", store_data.shape)
print("Test Data Shape:", test_data.shape)

### Step 5: Code execution

In [ ]:
# Display heads
order_data.head()


### Step 6: Code execution

In [ ]:
customer_data.head()

### Step 7: Code execution

In [ ]:
store_data.head()

### Step 8: Code execution

In [ ]:
test_data.head()

### Step 9: Display dataset information

In [ ]:
# Display info
print(order_data.info())

### Step 10: Display dataset information

In [ ]:
print(customer_data.info())

### Step 11: Display dataset information

In [ ]:
print(store_data.info())

### Step 12: Display dataset information

In [ ]:
print(test_data.info())

### Step 13: Check for missing values

In [ ]:
# Display missing values
print("🧩 Missing Values:")
print(order_data.isnull().sum())

### Step 14: Check for missing values

In [ ]:
print("🧩 Missing Values:")
print(customer_data.isnull().sum())

### Step 15: Check for missing values

In [ ]:
print("🧩 Missing Values:")
print(store_data.isnull().sum())

### Step 16: Check for missing values

In [ ]:
import textwrap

# DataFrames to analyze
datasets = {
    "order_data": order_data,
    "customer_data": customer_data,
    "store_data": store_data,
    "test_data": test_data
}

# Threshold for displaying unique values
unique_display_threshold = 36

# Column widths
col_widths = {
    "col_name": 30,
    "missing": 15,
    "unique": 15,
    "uniques_str": 60  # wrap width
}

# Function to print one row and wrap unique values nicely
def print_row(col, missing, unique_count, unique_str):
    col_fmt = f"{col:<{col_widths['col_name']}}"
    missing_fmt = f"{missing:<{col_widths['missing']}}"
    unique_fmt = f"{unique_count:<{col_widths['unique']}}"

    # Wrap the unique values to the allowed width
    wrapped_uniques = textwrap.wrap(unique_str, width=col_widths["uniques_str"])

    # First line: main row
    print(f"{col_fmt}{missing_fmt}{unique_fmt}{wrapped_uniques[0] if wrapped_uniques else ''}")

    # Additional lines: indented under the 4th column
    for line in wrapped_uniques[1:]:
        print(" " * (col_widths['col_name'] + col_widths['missing'] + col_widths['unique']) + line)

# Print per table
for name, df in datasets.items():
    print(f"\n==== {name} ====")
    header = f"{'Column Name':<{col_widths['col_name']}}{'Missing Values':<{col_widths['missing']}}{'Unique Count':<{col_widths['unique']}}Unique Values (if <39)"
    print(header)
    print("-" * len(header))

    for col in df.columns:
        missing = df[col].isnull().sum()
        unique_count = df[col].nunique(dropna=True)

        if unique_count < unique_display_threshold:
            uniques = df[col].dropna().unique()
            unique_str = ", ".join(map(str, uniques))
        else:
            unique_str = ""

        print_row(col, missing, unique_count, unique_str)

### Step 17: Parse and process ORDERS JSON data

In [ ]:
# Detect JSON columns
def detect_json_columns(df, df_name):
    json_cols = []
    for col in df.columns:
        sample = df[col].dropna().astype(str).iloc[0]  # take first non-null as string
        try:
            parsed = json.loads(sample)
            if isinstance(parsed, (dict, list)):
                json_cols.append(col)
        except:
            continue
    print(f"JSON-like columns in {df_name}: {json_cols if json_cols else 'None'}")

detect_json_columns(order_data, "order_data")
detect_json_columns(customer_data, "customer_data")
detect_json_columns(store_data, "store_data")
detect_json_columns(test_data, "test_data")

### Step 18: Parse and process ORDERS JSON data

In [ ]:
# 🔍 Extract and print all unique item_name values before generating wide_df
unique_item_names = set()

for idx, row in tqdm(order_data.iterrows(), total=len(order_data), desc='Extracting unique item_name values'):
    try:
        orders_field = row['ORDERS']
        if isinstance(orders_field, str):
            order_json = json.loads(orders_field)
        elif isinstance(orders_field, dict):
            order_json = orders_field
        else:
            continue

        item_details = order_json['orders'][0]['item_details']

        for item in item_details:
            item_name = item.get('item_name')
            if item_name:
                unique_item_names.add(item_name.strip())

    except Exception as e:
        print(f"Error in row {idx}: {e}")
        continue

# Print all unique item names
print(f"\n🔍 Found {len(unique_item_names)} unique item_name values:")
for name in sorted(unique_item_names):
    print(f"- {name}")

### Step 19: Code execution

In [ ]:
#checking if $19.99 Crispy Feast is an incorrect data where the prices column has come into the item_name column.

### Step 20: Parse and process ORDERS JSON data

In [ ]:
feast_prices = []

for idx, row in tqdm(order_data.iterrows(), total=len(order_data), desc='Checking item_price for "$19.99 Crispy Feast"'):
    try:
        orders_field = row['ORDERS']
        if isinstance(orders_field, str):
            order_json = json.loads(orders_field)
        elif isinstance(orders_field, dict):
            order_json = orders_field
        else:
            continue

        item_details = order_json['orders'][0]['item_details']

        for item in item_details:
            if item.get('item_name', '').strip() == "$19.99 Crispy Feast":
                feast_prices.append(float(item.get('item_price', 0)))

    except Exception as e:
        print(f"Error in row {idx}: {e}")
        continue

# Display results
print(f"\n🔍 Found {len(feast_prices)} entries for '$19.99 Crispy Feast'")
print("Unique item_price values for it:")
print(sorted(set(feast_prices)))

### Step 21: Parse and process ORDERS JSON data

In [ ]:
from tqdm import tqdm
import json

def parse_orders_to_columns(df):
    rows = []
    exclude_price = 0.0

    # Set of item names to ignore
    excluded_names = {
        'Delivery Fee',
        'Order Blankline 1',
        'Order Blankline 2',
        'Order Memo ASAP',
        'Order Memo Future',
        'Order Memo Item',
        'Order Memo Not Paid',
        'Order Memo Paid',
        'Plastic Fork',
        'Plastic Knife',
        'Plastic Straw',
        'Plastic Utensils',
        'Unavailable Item'
    }

    for idx, row in tqdm(df.iterrows(), total=len(df), desc='Parsing orders to columns'):
        try:
            orders_field = row['ORDERS']
            if isinstance(orders_field, str):
                order_json = json.loads(orders_field)
            elif isinstance(orders_field, dict):
                order_json = orders_field
            else:
                raise ValueError(f"Unexpected data type for ORDERS in row {idx}: {type(orders_field)}")

            item_details = order_json['orders'][0]['item_details']

            # Base order metadata
            base_data = {
                'CUSTOMER_ID': row['CUSTOMER_ID'],
                'STORE_NUMBER': row['STORE_NUMBER'],
                'ORDER_CREATED_DATE': row['ORDER_CREATED_DATE'],
                'ORDER_ID': row['ORDER_ID'],
                'ORDER_CHANNEL_NAME': row['ORDER_CHANNEL_NAME'],
                'ORDER_SUBCHANNEL_NAME': row['ORDER_SUBCHANNEL_NAME'],
                'ORDER_OCCASION_NAME': row['ORDER_OCCASION_NAME']
            }

            # Add each item as separate columns
            item_count = 0
            for item in item_details:
                name = item.get('item_name', '').strip()
                if name in excluded_names:
                    continue  # skip excluded item names
                item_count += 1
                base_data[f'item_name_{item_count}'] = name
                base_data[f'item_price_{item_count}'] = item['item_price']
                base_data[f'item_quantity_{item_count}'] = item['item_quantity']

            rows.append(base_data)

        except Exception as e:
            print(f"Error parsing row {idx}: {e}")
            continue

    return pd.DataFrame(rows)

### Step 22: Parse and process ORDERS JSON data

In [ ]:
wide_df = parse_orders_to_columns(order_data)

### Step 23: Code execution

In [ ]:
display(wide_df.head())

### Step 24: Code execution

In [ ]:
# Step 1: Identify all item_name_* columns
item_name_cols = [col for col in wide_df.columns if col.startswith('item_name_')]

# Step 2: Extract all values from those columns
all_item_names = pd.unique(wide_df[item_name_cols].values.ravel())

# Step 3: Remove NaN and sort
unique_item_names = sorted([name for name in all_item_names if pd.notna(name)])

# Step 4: Print
print(f"🔍 Found {len(unique_item_names)} unique item names across item_name_* columns:")
for name in unique_item_names:
    print(f"- {name}")

### Step 25: Code execution

In [ ]:
wide_df.count()

### Step 26: Code execution

In [ ]:
from IPython.display import display

display(wide_df[wide_df['item_quantity_15'].notna()])

### Step 27: Check for missing values

In [ ]:
print("🧩 Missing Values:")
print(customer_data.isnull().sum())

### Step 28: Display dataset information

In [ ]:
print("🔍 Data Info With Column Type")
print(customer_data.info())

### Step 29: Parse and process ORDERS JSON data

In [ ]:
# Step 1: Identify item quantity and item price columns
item_qty_cols = [col for col in wide_df.columns if col.startswith('item_quantity_')]
item_price_cols = [col for col in wide_df.columns if col.startswith('item_price_')]

# Step 2: Total items and total money per order (row)
wide_df['total_items_in_order'] = wide_df[item_qty_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
wide_df['total_money_spent'] = wide_df[item_price_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)

# Step 3: Aggregate per CUSTOMER_ID
customer_item_stats = wide_df.groupby('CUSTOMER_ID').agg(
    num_items_ordered=('total_items_in_order', 'sum'),
    total_money_spent=('total_money_spent', 'sum'),
    total_num_orders=('total_items_in_order', 'count')
).reset_index()

# Step 4: Merge with CUSTOMER_TYPE
customer_data_merged = customer_item_stats.merge(
    customer_data[['CUSTOMER_ID', 'CUSTOMER_TYPE']],
    on='CUSTOMER_ID',
    how='left'
)

# Optional: reorder columns
ordered_cols = [
    'CUSTOMER_ID', 'CUSTOMER_TYPE', 'total_num_orders',
    'num_items_ordered', 'total_money_spent'
]
customer_data_merged = customer_data_merged[ordered_cols]

### Step 30: Code execution

In [ ]:
display(customer_data_merged.head())

### Step 31: Code execution

In [ ]:
# Filter rows where CUSTOMER_TYPE is null (missing)
null_customer_type = customer_data_merged[customer_data_merged['CUSTOMER_TYPE'].isna()]

display(null_customer_type)

### Step 32: Parse and process ORDERS JSON data

In [ ]:
# Step 0: Add derived columns
customer_data_merged['avg_money_per_order'] = (
    customer_data_merged['total_money_spent'] / customer_data_merged['total_num_orders']
)

customer_data_merged['avg_money_per_item'] = (
    customer_data_merged['total_money_spent'] / customer_data_merged['num_items_ordered']
)

### Step 33: Code execution

In [ ]:
display(customer_data_merged.head())

### Step 34: Aggregate and compute summary metrics

In [ ]:
# Step 1: Filter out rows where CUSTOMER_TYPE is null or 'Deleted Account'
valid_customers = customer_data_merged[
    (customer_data_merged['CUSTOMER_TYPE'].notna()) &
    (customer_data_merged['CUSTOMER_TYPE'].str.strip() != 'Deleted Account')
]

# Step 2: Identify numeric columns including new ones
numeric_cols = [
    'avg_money_per_order',
    'avg_money_per_item'
]

# Step 3: Group by CUSTOMER_TYPE and calculate summary statistics
summary_stats = valid_customers.groupby('CUSTOMER_TYPE')[numeric_cols].agg(['mean'])

# Step 4: Display the result
display(summary_stats)

### Step 35: Merge datasets

In [ ]:
# Step 1: Identify customers with null CUSTOMER_TYPE
null_customers = customer_data_merged[
    customer_data_merged['CUSTOMER_TYPE'].isna()
][['CUSTOMER_ID', 'avg_money_per_order', 'avg_money_per_item']].copy()

# Step 2: Your fixed centroids
group_centroids = {
    'Guest': [32.52, 16.87],
    'Online': [38.63, 26.15],
    'Registered': [29.62, 15.19],
    'eClub': [33.04, 17.15]
}

# Step 3: Define function to assign closest group based on Euclidean distance
def assign_customer_type(row):
    min_dist = float('inf')
    assigned_type = None
    for group, (order_val, item_val) in group_centroids.items():
        dist = np.linalg.norm([
            row['avg_money_per_order'] - order_val,
            row['avg_money_per_item'] - item_val
        ])
        if dist < min_dist:
            min_dist = dist
            assigned_type = group
    return assigned_type

# Step 4: Apply assignment
null_customers['inferred_customer_type'] = null_customers.apply(assign_customer_type, axis=1)

# Step 5: Merge back to customer_data_merged
customer_data_merged = customer_data_merged.merge(
    null_customers[['CUSTOMER_ID', 'inferred_customer_type']],
    on='CUSTOMER_ID',
    how='left'
)

# Step 6: Fill null CUSTOMER_TYPE only
customer_data_merged['CUSTOMER_TYPE'] = customer_data_merged['CUSTOMER_TYPE'].fillna(
    customer_data_merged['inferred_customer_type']
)



### Step 36: Code execution

In [ ]:
customer_data_merged

### Step 37: Code execution

In [ ]:
print(customer_data_merged[customer_data_merged['inferred_customer_type'].notna()][
    ['CUSTOMER_ID', 'avg_money_per_order', 'avg_money_per_item', 'CUSTOMER_TYPE', 'inferred_customer_type']
])

### Step 38: Code execution

In [ ]:
# Step 7: Drop helper column
customer_data_merged.drop(columns=['inferred_customer_type'], inplace=True)

### Step 39: Inspect unique values in columns

In [ ]:
# Unique elements
unique_types = customer_data_merged['CUSTOMER_TYPE'].unique()
print("Unique CUSTOMER_TYPE values:")
print(unique_types)

# Count of each type
type_counts = customer_data_merged['CUSTOMER_TYPE'].value_counts()
print("\nCount of each CUSTOMER_TYPE:")
print(type_counts)

### Step 40: Code execution

In [ ]:
# Original CUSTOMER_TYPE counts from customer_data before any inference
original_type_counts = customer_data['CUSTOMER_TYPE'].value_counts(dropna=False)

print("📊 Original CUSTOMER_TYPE counts (before filling nulls):")
print(original_type_counts)

### Step 41: Inspect unique values in columns

In [ ]:
order_data['CUSTOMER_ID'].nunique()

# Store

### Step 42: Check for missing values

In [ ]:
print("\n🧩 Missing Values:")
print(store_data.isnull().sum())

### Step 43: Display dataset information

In [ ]:
print("🔍 Data Info With Column Type")
print(store_data.info())

### Step 44: Code execution

In [ ]:
display(store_data)

# Merging df

### Step 45: Merge datasets

In [ ]:
# Step 1: Copy wide_df
merged_dataset = wide_df.copy()

# Step 2: Merge CUSTOMER_TYPE from customer_data_merged
merged_dataset = merged_dataset.merge(
    customer_data_merged[['CUSTOMER_ID', 'CUSTOMER_TYPE']],
    on='CUSTOMER_ID',
    how='left'
)

# Optional: View the result
print("✅ Merged dataset with CUSTOMER_TYPE added:")
display(merged_dataset.head())

### Step 46: Code execution

In [ ]:
print("📊 Count of unique CUSTOMER_TYPE values in merged_dataset:")
print(merged_dataset['CUSTOMER_TYPE'].value_counts())

### Step 47: Code execution

In [ ]:
merged_dataset = merged_dataset.drop(columns=['ORDER_CREATED_DATE'])

### Step 48: Check for missing values

In [ ]:
import textwrap

# DataFrames to analyze
datasets = {
    "merged_dataset": merged_dataset,
    "test_data": test_data
}

# Threshold for displaying unique values
unique_display_threshold = 36

# Column widths
col_widths = {
    "col_name": 30,
    "missing": 15,
    "unique": 15,
    "uniques_str": 60  # wrap width
}

# Function to print one row and wrap unique values nicely
def print_row(col, missing, unique_count, unique_str):
    col_fmt = f"{col:<{col_widths['col_name']}}"
    missing_fmt = f"{missing:<{col_widths['missing']}}"
    unique_fmt = f"{unique_count:<{col_widths['unique']}}"

    # Wrap the unique values to the allowed width
    wrapped_uniques = textwrap.wrap(unique_str, width=col_widths["uniques_str"])

    # First line: main row
    print(f"{col_fmt}{missing_fmt}{unique_fmt}{wrapped_uniques[0] if wrapped_uniques else ''}")

    # Additional lines: indented under the 4th column
    for line in wrapped_uniques[1:]:
        print(" " * (col_widths['col_name'] + col_widths['missing'] + col_widths['unique']) + line)

# Print per table
for name, df in datasets.items():
    print(f"\n==== {name} ====")
    header = f"{'Column Name':<{col_widths['col_name']}}{'Missing Values':<{col_widths['missing']}}{'Unique Count':<{col_widths['unique']}}Unique Values (if <{unique_display_threshold})"
    print(header)
    print("-" * len(header))

    for col in df.columns:
        missing = df[col].isnull().sum()
        unique_count = df[col].nunique(dropna=True)

        if unique_count < unique_display_threshold:
            uniques = df[col].dropna().unique()
            unique_str = ", ".join(map(str, uniques))
        else:
            unique_str = ""

        print_row(col, missing, unique_count, unique_str)

### Step 49: Code execution

In [ ]:
# Create merged_dataset_final by dropping unwanted columns
merged_dataset_final = merged_dataset.drop(
    columns=[col for col in merged_dataset.columns if
             col.startswith('item_price_') or
             col.startswith('item_quantity_') or
             col == 'ORDER_CHANNEL_NAME' or
             col == 'total_money_spent' or
             col == 'total_items_in_order']
)

# Check the resulting table
merged_dataset_final.head()

### Step 50: Check for missing values

In [ ]:
import textwrap

# DataFrames to analyze
datasets = {
    "merged_dataset_final": merged_dataset_final,
    "test_data": test_data
}

# Threshold for displaying unique values
unique_display_threshold = 6

# Column widths
col_widths = {
    "col_name": 30,
    "missing": 15,
    "unique": 15,
    "uniques_str": 60  # wrap width
}

# Function to print one row and wrap unique values nicely
def print_row(col, missing, unique_count, unique_str):
    col_fmt = f"{col:<{col_widths['col_name']}}"
    missing_fmt = f"{missing:<{col_widths['missing']}}"
    unique_fmt = f"{unique_count:<{col_widths['unique']}}"

    # Wrap the unique values to the allowed width
    wrapped_uniques = textwrap.wrap(unique_str, width=col_widths["uniques_str"])

    # First line: main row
    print(f"{col_fmt}{missing_fmt}{unique_fmt}{wrapped_uniques[0] if wrapped_uniques else ''}")

    # Additional lines: indented under the 4th column
    for line in wrapped_uniques[1:]:
        print(" " * (col_widths['col_name'] + col_widths['missing'] + col_widths['unique']) + line)

# Print per table
for name, df in datasets.items():
    print(f"\n==== {name} ====")
    header = f"{'Column Name':<{col_widths['col_name']}}{'Missing Values':<{col_widths['missing']}}{'Unique Count':<{col_widths['unique']}}Unique Values (if <{unique_display_threshold})"
    print(header)
    print("-" * len(header))

    for col in df.columns:
        missing = df[col].isnull().sum()
        unique_count = df[col].nunique(dropna=True)

        if unique_count < unique_display_threshold:
            uniques = df[col].dropna().unique()
            unique_str = ", ".join(map(str, uniques))
        else:
            unique_str = ""

        print_row(col, missing, unique_count, unique_str)

### Step 51: Code execution

In [ ]:
# Step 1: Rename item_name_* columns to item1, item2, ...
rename_map = {}
for i in range(1, 16):
    old_name = f'item_name_{i}'
    new_name = f'item{i}'
    if old_name in merged_dataset_final.columns:
        rename_map[old_name] = new_name

merged_dataset_final = merged_dataset_final.rename(columns=rename_map)

# Step 2: Move CUSTOMER_TYPE to 4th column position
cols = list(merged_dataset_final.columns)
if 'CUSTOMER_TYPE' in cols:
    cols.insert(3, cols.pop(cols.index('CUSTOMER_TYPE')))
    merged_dataset_final = merged_dataset_final[cols]

# Step 3: Check result
merged_dataset_final.head()

### Step 52: Check for missing values

In [ ]:
import textwrap

# DataFrames to analyze
datasets = {
    "merged_dataset_final": merged_dataset_final,
    "test_data": test_data
}

# Threshold for displaying unique values
unique_display_threshold = 6

# Column widths
col_widths = {
    "col_name": 30,
    "missing": 15,
    "unique": 15,
    "uniques_str": 60  # wrap width
}

# Function to print one row and wrap unique values nicely
def print_row(col, missing, unique_count, unique_str):
    col_fmt = f"{col:<{col_widths['col_name']}}"
    missing_fmt = f"{missing:<{col_widths['missing']}}"
    unique_fmt = f"{unique_count:<{col_widths['unique']}}"

    # Wrap the unique values to the allowed width
    wrapped_uniques = textwrap.wrap(unique_str, width=col_widths["uniques_str"])

    # First line: main row
    print(f"{col_fmt}{missing_fmt}{unique_fmt}{wrapped_uniques[0] if wrapped_uniques else ''}")

    # Additional lines: indented under the 4th column
    for line in wrapped_uniques[1:]:
        print(" " * (col_widths['col_name'] + col_widths['missing'] + col_widths['unique']) + line)

# Print per table
for name, df in datasets.items():
    print(f"\n==== {name} ====")
    header = f"{'Column Name':<{col_widths['col_name']}}{'Missing Values':<{col_widths['missing']}}{'Unique Count':<{col_widths['unique']}}Unique Values (if <{unique_display_threshold})"
    print(header)
    print("-" * len(header))

    for col in df.columns:
        missing = df[col].isnull().sum()
        unique_count = df[col].nunique(dropna=True)

        if unique_count < unique_display_threshold:
            uniques = df[col].dropna().unique()
            unique_str = ", ".join(map(str, uniques))
        else:
            unique_str = ""

        print_row(col, missing, unique_count, unique_str)

### Step 53: Code execution

In [ ]:
# Step 1: Remove rows where CUSTOMER_TYPE is 'Online' or 'Deleted Account'
merged_dataset_final = merged_dataset_final[
    ~merged_dataset_final['CUSTOMER_TYPE'].isin(['Online', 'Deleted Account'])
]

# Step 2: Remove rows where ORDER_SUBCHANNEL_NAME is 'Grub Hub Marketplace'
merged_dataset_final = merged_dataset_final[
    merged_dataset_final['ORDER_SUBCHANNEL_NAME'] != 'Grub Hub Marketplace'
]

# Step 3: Reset index after deletions
merged_dataset_final = merged_dataset_final.reset_index(drop=True)

### Step 54: Code execution

In [ ]:
# Drop the ORDER_SUBCHANNEL_NAME column
merged_dataset_final = merged_dataset_final.drop(columns=['ORDER_SUBCHANNEL_NAME'])

### Step 55: Check for missing values

In [ ]:
# DataFrames to analyze
datasets = {
    "merged_dataset_final": merged_dataset_final,
    "test_data": test_data
}

# Threshold for displaying unique values
unique_display_threshold = 6

# Column widths
col_widths = {
    "col_name": 30,
    "missing": 15,
    "unique": 15,
    "uniques_str": 60  # wrap width
}

# Function to print one row and wrap unique values nicely
def print_row(col, missing, unique_count, unique_str):
    col_fmt = f"{col:<{col_widths['col_name']}}"
    missing_fmt = f"{missing:<{col_widths['missing']}}"
    unique_fmt = f"{unique_count:<{col_widths['unique']}}"

    # Wrap the unique values to the allowed width
    wrapped_uniques = textwrap.wrap(unique_str, width=col_widths["uniques_str"])

    # First line: main row
    print(f"{col_fmt}{missing_fmt}{unique_fmt}{wrapped_uniques[0] if wrapped_uniques else ''}")

    # Additional lines: indented under the 4th column
    for line in wrapped_uniques[1:]:
        print(" " * (col_widths['col_name'] + col_widths['missing'] + col_widths['unique']) + line)

# Print per table
for name, df in datasets.items():
    print(f"\n==== {name} ====")
    header = f"{'Column Name':<{col_widths['col_name']}}{'Missing Values':<{col_widths['missing']}}{'Unique Count':<{col_widths['unique']}}Unique Values (if <{unique_display_threshold})"
    print(header)
    print("-" * len(header))

    for col in df.columns:
        missing = df[col].isnull().sum()
        unique_count = df[col].nunique(dropna=True)

        if unique_count < unique_display_threshold:
            uniques = df[col].dropna().unique()
            unique_str = ", ".join(map(str, uniques))
        else:
            unique_str = ""

        print_row(col, missing, unique_count, unique_str)

### Step 56: Code execution

In [ ]:
# Find common ORDER_IDs
common_order_ids = set(test_data['ORDER_ID']).intersection(set(merged_dataset_final['ORDER_ID']))

# Print result
if common_order_ids:
    print(f"Number of common ORDER_IDs: {len(common_order_ids)}")
    print("Sample common ORDER_IDs:", list(common_order_ids)[:10])
else:
    print("No common ORDER_IDs found.")

### Step 57: Code execution

In [ ]:
merged_dataset_final = merged_dataset_final.drop(columns=['ORDER_ID'])

### Step 58: Code execution

In [ ]:
merged_dataset_final.head()

### Step 59: Check for missing values

In [ ]:
# DataFrames to analyze
datasets = {
    "merged_dataset_final": merged_dataset_final,
    "test_data": test_data
}

# Threshold for displaying unique values
unique_display_threshold = 6

# Column widths
col_widths = {
    "col_name": 30,
    "missing": 15,
    "unique": 15,
    "uniques_str": 60  # wrap width
}

# Function to print one row and wrap unique values nicely
def print_row(col, missing, unique_count, unique_str):
    col_fmt = f"{col:<{col_widths['col_name']}}"
    missing_fmt = f"{missing:<{col_widths['missing']}}"
    unique_fmt = f"{unique_count:<{col_widths['unique']}}"

    # Wrap the unique values to the allowed width
    wrapped_uniques = textwrap.wrap(unique_str, width=col_widths["uniques_str"])

    # First line: main row
    print(f"{col_fmt}{missing_fmt}{unique_fmt}{wrapped_uniques[0] if wrapped_uniques else ''}")

    # Additional lines: indented under the 4th column
    for line in wrapped_uniques[1:]:
        print(" " * (col_widths['col_name'] + col_widths['missing'] + col_widths['unique']) + line)

# Print per table
for name, df in datasets.items():
    print(f"\n==== {name} ====")
    header = f"{'Column Name':<{col_widths['col_name']}}{'Missing Values':<{col_widths['missing']}}{'Unique Count':<{col_widths['unique']}}Unique Values (if <{unique_display_threshold})"
    print(header)
    print("-" * len(header))

    for col in df.columns:
        missing = df[col].isnull().sum()
        unique_count = df[col].nunique(dropna=True)

        if unique_count < unique_display_threshold:
            uniques = df[col].dropna().unique()
            unique_str = ", ".join(map(str, uniques))
        else:
            unique_str = ""

        print_row(col, missing, unique_count, unique_str)

### Step 60: Code execution

In [ ]:
# Count rows for each unique value in CUSTOMER_TYPE (excluding nulls)
customer_type_counts = merged_dataset_final['CUSTOMER_TYPE'].value_counts(dropna=True)

# Count rows for each unique value in ORDER_OCCASION_NAME (excluding nulls)
order_occasion_counts = merged_dataset_final['ORDER_OCCASION_NAME'].value_counts(dropna=True)

# Display the results
print("Counts for CUSTOMER_TYPE:")
print(customer_type_counts)
print("\nCounts for ORDER_OCCASION_NAME:")
print(order_occasion_counts)

### Step 61: Create visualization

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Step 1: Create numerical copy for correlation
corr_data = merged_dataset_final.copy()

# Convert categorical columns to numerical using label encoding
corr_data = corr_data.apply(lambda col: col.astype('category').cat.codes
                            if col.dtypes == 'object' else col)

# Step 2: Calculate correlation matrix
corr_matrix = corr_data.corr()

# Assume corr_matrix is already calculated
for i in range(19):  # 19 columns
    corr_matrix.iat[i, i] = 0.0

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Heatmap (Diagonal = 0)")
plt.show()

### Step 62: Create visualization

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Step 1: Create numerical copy for correlation
corr_data = merged_dataset_final.copy()

# Convert categorical columns to numerical using label encoding
corr_data = corr_data.apply(lambda col: col.astype('category').cat.codes
                            if col.dtypes == 'object' else col)

# Step 2: Calculate correlation matrix
corr_matrix = corr_data.corr()

# --- Output correlation matrix separately ---
print("Correlation Matrix:")
display(corr_matrix)

### Step 63: Display dataset information

In [ ]:
merged_dataset_final.info()

### Step 64: Code execution

In [ ]:
#pip install mlxtend

### Step 65: Modelling Data With FP-Growth Model and Validating With RECALL@3

In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
from tqdm import tqdm
import random
import numpy as np
from itertools import combinations

tqdm.pandas()

# ----------------
# Prepare items column if not already
if 'items' not in merged_dataset_final.columns:
    item_cols = [f"item{i}" for i in range(1, 16)]
    merged_dataset_final['items'] = merged_dataset_final[item_cols].progress_apply(
        lambda x: [i for i in x if pd.notnull(i)],
        axis=1
    )

# Shuffle and split
merged_dataset_final = merged_dataset_final.sample(frac=1, random_state=42).reset_index(drop=True)
train_split, val_split = 0.7, 0.15
n = len(merged_dataset_final)
train_df = merged_dataset_final.iloc[:int(train_split*n)]
val_df = merged_dataset_final.iloc[int(train_split*n):int((train_split+val_split)*n)]
test_df = merged_dataset_final.iloc[int((train_split+val_split)*n):]

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Transaction encoding on train
print("\nEncoding transactions...")
te = TransactionEncoder()
te_ary = list(tqdm(te.fit(train_df['items']).transform(train_df['items']),
                   total=len(train_df),
                   desc="Transaction Encoding"))
basket_df = pd.DataFrame(te_ary, columns=te.columns_)

# Run FP-Growth
print("\nRunning FP-Growth...")
frequent_itemsets = fpgrowth(basket_df, min_support=0.001, use_colnames=True)
print(f"Found {len(frequent_itemsets)} frequent itemsets")

# Generate association rules
print("\nGenerating association rules...")
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.1)
rules = rules.sort_values(by='confidence', ascending=False)
print(f"Generated {len(rules)} association rules")

# Prediction function
def suggest_next_items(selected_items, rules, top_n=3):
    selected_items = set(selected_items)
    all_suggestions = {}

    for _, rule in rules.iterrows():
        if rule['antecedents'].issubset(selected_items):
            for consequent in rule['consequents']:
                if consequent not in selected_items:
                    all_suggestions[consequent] = max(all_suggestions.get(consequent, 0), rule['confidence'])

    sorted_suggestions = sorted(all_suggestions.items(), key=lambda item: item[1], reverse=True)
    return [item[0] for item in sorted_suggestions][:top_n]

# Sample validation suggestions for all 3-item combinations
sample_order = random.choice(val_df['items'].tolist())
if len(sample_order) >= 3:
    print("\nValidation suggestions for all possible 3-item combinations from the sample order:")
    for combo in tqdm(list(combinations(sample_order, 3)), desc="Generating sample predictions"):
        given_items = set(combo)
        suggestions = suggest_next_items(given_items, rules, top_n=3)
        print(f"Given items: {list(given_items)} --> Suggested next: {suggestions}")
else:
    print("Sample order had less than 3 items, skipping.")

# Evaluate on test set
print("\nEvaluating on test set...")
test_items_list = test_df['items'].tolist()

hits, cases = 0, 0
min_items_for_eval = 4  # Need at least 4 items for given 3 + actual

for items in tqdm(test_items_list, desc="Evaluating hit rate"):
    if len(items) < min_items_for_eval:
        continue
    given_items = set(items[:3])
    actual_items = set(items[3:])
    predicted_items = set(suggest_next_items(given_items, rules, top_n=3))
    if predicted_items & actual_items:
        hits += 1
    cases += 1

hit_rate = hits / cases if cases > 0 else 0
print(f"\nHit rate@3 (given 3 items): {hit_rate:.2%}")

# Predicting - Test Dataset

In [ ]:
# Add empty columns
test_data['item4'] = np.nan
test_data['RECOMMENDATION 1'] = np.nan
test_data['RECOMMENDATION 2'] = np.nan
test_data['RECOMMENDATION 3'] = np.nan

def get_recommendations(row):
    items = [row['item1'], row['item2'], row['item3']]
    items = [i for i in items if pd.notna(i) and i != '']
    recs = suggest_next_items(items, rules, top_n=3)
    recs += [np.nan] * (3 - len(recs))
    return pd.Series(recs[:3])

# Apply and fill recommendations
test_data[['RECOMMENDATION 1', 'RECOMMENDATION 2', 'RECOMMENDATION 3']] = test_data.progress_apply(get_recommendations, axis=1)

In [ ]:
display(test_data)

In [ ]:
filename = "DisCovert_Recommendation Output Sheet.xlsx"
test_data.to_excel(filename, index=False)
print(f"Saved test_data to Excel file: {filename}")